In [5]:
import wandb
import pandas as pd
from scipy import stats

api = wandb.Api()

# 방법 1: sweep_id로 직접 가져오기 (권장)
sweep = api.sweep("jjeong3150-aiffel/peekabook-crs-test3/z7rm3pvq")
runs = sweep.runs

# 방법 2: 프로젝트 전체에서 sweep_id로 필터링
# runs = api.runs("YOUR_ENTITY/YOUR_PROJECT", 
#                 filters={"sweep": "YOUR_SWEEP_ID"})

records = []
for run in runs:
    records.append({
        "collection":       run.config.get("collection_name"),
        "use_genre_filter": run.config.get("use_genre_filter"),
        "query_transform":  run.config.get("query_transform"),
        "persona":          run.config.get("persona_name"),
        "run_index":        run.config.get("run_index"),
        "mean_score":       run.summary.get("mean_score"),
    })

df = pd.DataFrame(records).dropna(subset=["mean_score"])
print(f"총 runs: {len(df)}")  # 72개인지 확인
print(df.groupby("collection")["mean_score"].describe())

# 컬렉션 간 비교
intro  = df[df.collection == "books_intro_48k"]["mean_score"]
merged = df[df.collection == "books_merged_48k"]["mean_score"]
t, p = stats.ttest_ind(intro, merged)
print(f"\nintro vs merged: t={t:.3f}, p={p:.3f}")

# 필터 효과 비교
filt_true  = df[df.use_genre_filter == True]["mean_score"]
filt_false = df[df.use_genre_filter == False]["mean_score"]
t2, p2 = stats.ttest_ind(filt_true, filt_false)
print(f"filter=T vs F:   t={t2:.3f}, p={p2:.3f}")

총 runs: 59
                  count      mean       std  min       25%       50%  \
collection                                                             
books_merged_48k   59.0  0.463277  0.239867  0.0  0.333333  0.333333   

                       75%  max  
collection                       
books_merged_48k  0.666667  1.0  

intro vs merged: t=nan, p=nan
filter=T vs F:   t=-0.469, p=0.641


/tmp/ipykernel_90283/3107326940.py:33: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  t, p = stats.ttest_ind(intro, merged)


In [4]:
# 컬렉션별 필터 효과 분리 검정
for col in ["books_intro_48k", "books_merged_48k"]:
    subset = df[df.collection == col]
    
    filt_true  = subset[subset.use_genre_filter == True]["mean_score"]
    filt_false = subset[subset.use_genre_filter == False]["mean_score"]
    
    t, p = stats.ttest_ind(filt_true, filt_false)
    print(f"\n[{col}]")
    print(f"  filter=True  mean: {filt_true.mean():.3f}, std: {filt_true.std():.3f}, n={len(filt_true)}")
    print(f"  filter=False mean: {filt_false.mean():.3f}, std: {filt_false.std():.3f}, n={len(filt_false)}")
    print(f"  t={t:.3f}, p={p:.3f}")


[books_intro_48k]
  filter=True  mean: 0.593, std: 0.293, n=18
  filter=False mean: 0.463, std: 0.203, n=18
  t=1.545, p=0.132

[books_merged_48k]
  filter=True  mean: 0.500, std: 0.171, n=18
  filter=False mean: 0.519, std: 0.235, n=18
  t=-0.270, p=0.789


In [12]:
import wandb
import pandas as pd
from scipy import stats

api = wandb.Api()

# 모든 sweep 합치기
sweep_ids = ["48z8l79r", "pecidrs2", "z7rm3pvq"]  # 실제 ID로 교체

records = []
for sweep_id in sweep_ids:
    sweep = api.sweep(f"jjeong3150-aiffel/peekabook-crs-test3/{sweep_id}")
    for run in sweep.runs:
        records.append({
            "collection":       run.config.get("collection_name"),
            "use_genre_filter": run.config.get("use_genre_filter"),
            "query_transform":  run.config.get("query_transform"),
            "persona":          run.config.get("persona_name"),
            "run_index":        run.config.get("run_index"),
            "mean_score":       run.summary.get("mean_score"),
        })

df = pd.DataFrame(records).dropna(subset=["mean_score"])

# books_merged_48k만 필터링
merged = df[df.collection == "books_merged_48k"]
print(f"merged runs: {len(merged)}")

# 필터 효과 t-test
filt_true  = merged[merged.use_genre_filter == True]["mean_score"]
filt_false = merged[merged.use_genre_filter == False]["mean_score"]

t, p = stats.ttest_ind(filt_true, filt_false)
print(f"\nfilter=True  mean: {filt_true.mean():.3f}, std: {filt_true.std():.3f}, n={len(filt_true)}")
print(f"filter=False mean: {filt_false.mean():.3f}, std: {filt_false.std():.3f}, n={len(filt_false)}")
print(f"t={t:.3f}, p={p:.3f}")

merged runs: 112

filter=True  mean: 0.476, std: 0.228, n=56
filter=False mean: 0.518, std: 0.219, n=56
t=-0.985, p=0.327


## books_merged_48k에 필터링 유무 차이 확인

In [13]:
# books_merged_48k만 필터링
merged = df[df.collection == "books_intro_48k"]
print(f"merged runs: {len(merged)}")

# 필터 효과 t-test
filt_true  = merged[merged.use_genre_filter == True]["mean_score"]
filt_false = merged[merged.use_genre_filter == False]["mean_score"]

t, p = stats.ttest_ind(filt_true, filt_false)
print(f"\nfilter=True  mean: {filt_true.mean():.3f}, std: {filt_true.std():.3f}, n={len(filt_true)}")
print(f"filter=False mean: {filt_false.mean():.3f}, std: {filt_false.std():.3f}, n={len(filt_false)}")
print(f"t={t:.3f}, p={p:.3f}")

merged runs: 130

filter=True  mean: 0.562, std: 0.271, n=64
filter=False mean: 0.495, std: 0.236, n=66
t=1.517, p=0.132


In [16]:
import pandas as pd
from scipy import stats

# books_intro_48k만 필터링
intro = df[df.collection == "books_intro_48k"]
print(f"intro runs: {len(intro)}")

# 쿼리변환 효과 t-test
qt_rewrite = intro[intro.query_transform == "rewrite_decompose"]["mean_score"]
qt_none    = intro[intro.query_transform == "none"]["mean_score"]

t, p = stats.ttest_ind(qt_rewrite, qt_none)

# 그룹별 표
result_df = pd.DataFrame([
    {"조건": "rewrite_decompose", "mean": f"{qt_rewrite.mean():.3f}", "std": f"{qt_rewrite.std():.3f}", "n": len(qt_rewrite)},
    {"조건": "none",              "mean": f"{qt_none.mean():.3f}",    "std": f"{qt_none.std():.3f}",    "n": len(qt_none)},
])

from IPython.display import display
display(result_df.style.hide(axis="index"))

# t, p 따로 출력
print(f"t={t:.3f}, p={p:.3f}")

intro runs: 130


조건,mean,std,n
rewrite_decompose,0.508,0.271,65
none,0.549,0.239,65


t=-0.916, p=0.361


In [17]:
# run 하나 꺼내서 어떤 필드가 있는지 확인
sample_run = next(iter(sweep.runs))
print(sample_run.summary.keys())
print(sample_run.config.keys())

dict_keys(['_runtime', '_step', '_timestamp', '_wandb', 'book_1_score', 'book_2_score', 'book_3_score', 'chroma_db_path', 'collection_name', 'mean_score', 'persona', 'persona_name', 'query_transform', 'reflection', 'results_table', 'summary', 'use_genre_filter'])
dict_keys(['run_index', 'persona_name', 'collection_name', 'query_transform', 'use_genre_filter'])


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

api = wandb.Api()
sweep_ids = ["48z8l79r", "pecidrs2", "z7rm3pvq"]

records = []
for sweep_id in sweep_ids:
    sweep = api.sweep(f"jjeong3150-aiffel/peekabook-crs-test3/{sweep_id}")
    for run in sweep.runs:
        b1 = run.summary.get("book_1_score", 0)
        b2 = run.summary.get("book_2_score", 0)
        b3 = run.summary.get("book_3_score", 0)
        
        # NDCG@3 계산
        relevance = [b1, b2, b3]
        dcg   = sum(rel / np.log2(rank + 1) 
                    for rank, rel in enumerate(relevance, start=1))
        ideal = sum(1 / np.log2(rank + 1) 
                    for rank in range(1, 4))
        ndcg  = dcg / ideal if ideal > 0 else 0
        
        records.append({
            "collection":       run.config.get("collection_name"),
            "use_genre_filter": run.config.get("use_genre_filter"),
            "query_transform":  run.config.get("query_transform"),
            "persona":          run.config.get("persona_name"),
            "run_index":        run.config.get("run_index"),
            "mean_score":       run.summary.get("mean_score"),
            "ndcg3":            ndcg,
        })

df = pd.DataFrame(records).dropna(subset=["mean_score"])
display(df[["collection", "use_genre_filter", "query_transform", "mean_score", "ndcg3"]].groupby(
    ["collection", "use_genre_filter", "query_transform"]
).mean().round(3))

mean_score  ndcg3
collection       use_genre_filter query_transform                     
books_intro_48k  False            none                    0.515  0.563
                                  rewrite_decompose       0.475  0.492
                 True             none                    0.583  0.598
                                  rewrite_decompose       0.542  0.575
books_merged_48k False            none                    0.511  0.488
                                  rewrite_decompose       0.556  0.588
                 True             none                    0.471  0.467
                                  rewrite_decompose       0.500  0.530

In [21]:
display(df[["collection", "use_genre_filter", "query_transform", "mean_score", "ndcg3"]]
    .groupby(["collection", "use_genre_filter", "query_transform"])
    .agg(
        mean_score_count=("mean_score", "count"),
        mean_score_mean=("mean_score", "mean"),
        ndcg3_mean=("ndcg3", "mean"),
    )
    .round(3)
)

mean_score_count  \
collection       use_genre_filter query_transform                       
books_intro_48k  False            none                             33   
                                  rewrite_decompose                33   
                 True             none                             32   
                                  rewrite_decompose                32   
books_merged_48k False            none                             47   
                                  rewrite_decompose                 9   
                 True             none                             46   
                                  rewrite_decompose                10   

                                                     mean_score_mean  \
collection       use_genre_filter query_transform                      
books_intro_48k  False            none                         0.515   
                                  rewrite_decompose            0.475   
                 True             none                         0.583   
                                  rewrite_decompose            0.542   
books_merged_48k False            none                         0.511   
                                  rewrite_decompose            0.556   
                 True             none                         0.471   
                                  rewrite_decompose            0.500   

                                                     ndcg3_mean  
collection       use_genre_filter query_transform                
books_intro_48k  False            none                    0.563  
                                  rewrite_decompose       0.492  
                 True             none                    0.598  
                                  rewrite_decompose       0.575  
books_merged_48k False            none                    0.488  
                                  rewrite_decompose       0.588  
                 True             none                    0.467  
                                  rewrite_decompose       0.530

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

api = wandb.Api()
sweep_ids = ["48z8l79r", "pecidrs2", "z7rm3pvq"]

records = []
for sweep_id in sweep_ids:
    sweep = api.sweep(f"jjeong3150-aiffel/peekabook-crs-test3/{sweep_id}")
    for run in sweep.runs:
        b1 = run.summary.get("book_1_score", 0)
        b2 = run.summary.get("book_2_score", 0)
        b3 = run.summary.get("book_3_score", 0)
        
        # NDCG@3 계산
        relevance = [b1, b2, b3]
        dcg   = sum(rel / np.log2(rank + 1) 
                    for rank, rel in enumerate(relevance, start=1))
        ideal = sum(1 / np.log2(rank + 1) 
                    for rank in range(1, 4))
        ndcg  = dcg / ideal if ideal > 0 else 0
        
        # Hit Rate@3 계산 (1권 이상 적절하면 1.0)
        hit3 = 1.0 if any(s > 0 for s in [b1, b2, b3]) else 0.0
        
        records.append({
            "collection":       run.config.get("collection_name"),
            "use_genre_filter": run.config.get("use_genre_filter"),
            "query_transform":  run.config.get("query_transform"),
            "persona":          run.config.get("persona_name"),
            "run_index":        run.config.get("run_index"),
            "mean_score":       run.summary.get("mean_score"),
            "ndcg3":            ndcg,
            "hit_rate3":        hit3,
        })

df = pd.DataFrame(records).dropna(subset=["mean_score"])
display(df[["collection", "use_genre_filter", "query_transform", "mean_score", "ndcg3", "hit_rate3"]].groupby(
    ["collection", "use_genre_filter", "query_transform"]
).mean().round(3))